# THANTD — Colab Training Notebook
**Triplet Hybrid Attention Network for Hyperspectral Target Detection**  
Liu et al., IEEE JSTARS 2025

### What this notebook does
1. Mounts your Google Drive  
2. Installs the `hyspan` library from GitHub  
3. Loads 2–3 real HSI datasets from `MyDrive/hyspan_datasets/`  
4. Per dataset: synthesises a **(500 × 250)** HSI using GMM-based synthesis  
5. Splits into **train (350 × 250)** and **test (150 × 250)**  
6. Trains THANTD on the training split  
7. Evaluates on the test split (AUC + detection map)  
8. Saves synthetic images, checkpoints, and results to Drive

### Drive layout expected
```
MyDrive/
  hyspan_datasets/
    Sandiego.mat          # key: 'data' (H,W,D), 'map' (H,W)
    abu-airport-2.mat
    (pavia-u.mat)         # optional — very large
```
Each `.mat` file should contain at minimum a `data` field with the HSI array
and a `map` field with the binary ground-truth (1=target, 0=background).

## 0 · Runtime check

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cpu':
    print('WARNING: No GPU detected. Go to Runtime → Change runtime type → GPU.')

## 1 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT    = '/content/drive/MyDrive'
DATASET_DIR   = os.path.join(DRIVE_ROOT, 'hyspan_datasets')
OUTPUT_DIR    = os.path.join(DRIVE_ROOT, 'hyspan_outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Datasets:', os.listdir(DATASET_DIR))

## 2 · Install hyspan

In [ ]:
!pip install -q git+https://github.com/michaelpiro/hyspan.git
# Reload site-packages in case of first install
import importlib, site
importlib.invalidate_caches()

## 3 · Imports

In [ ]:
import os
import math
import json
import numpy as np
import scipy.io as sio
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from hyspan.deep_models.THANTD import (
    THANTD, ETBLoss, TripletDataset,
    build_dataset_from_image, train_thantd, thantd_detect,
)
from hyspan.synthesis     import SynthesisEngine
from hyspan.results       import roc_auc
from hyspan.algorithms.utils import ts_generation

print('hyspan imported OK')

## 4 · Configuration

In [ ]:
# ── Dataset files to process (must exist in DATASET_DIR) ──────────────────
DATASET_FILES = [
    'Sandiego.mat',
    'abu-airport-2.mat',
    # 'pavia-u.mat',   # uncomment if you have it — very large, needs lots of RAM
]

# ── Synthesis settings ────────────────────────────────────────────────────
SYNTH_H        = 500    # full synthetic image height
SYNTH_W        = 250    # full synthetic image width
TRAIN_H        = 350    # rows kept for training
TEST_H         = 150    # rows kept for testing  (TRAIN_H + TEST_H == SYNTH_H)
N_GMM_COMPS    = 10     # GMM components for synthesis
PCA_COMPONENTS = 30     # reduce to this many bands before training
                         # (set to None to skip PCA)

# ── THANTD model ──────────────────────────────────────────────────────────
MODEL_CFG = dict(
    d_model       = 64,
    n_heads       = 4,
    n_layers      = 2,
    window_size   = 3,
    mlp_ratio     = 4,
    cam_reduction = 4,
    dropout       = 0.1,
)

# ── Training ──────────────────────────────────────────────────────────────
TRAIN_CFG = dict(
    epochs       = 60,
    batch_size   = 512,
    lr           = 1e-4,
    weight_decay = 1e-4,
    margin       = 0.3,
    lambda_bce   = 0.5,
    n_triplets   = 60_000,
    device       = device,
    print_every  = 10,
)

## 5 · Helper utilities

In [ ]:
def load_mat(path):
    """Load a .mat file and return (image, gt) as float32 numpy arrays."""
    mat  = sio.loadmat(path)
    # Try common key names for the HSI data
    for key in ['data', 'Data', 'img', 'image', 'HSI', 'hsi']:
        if key in mat:
            img = mat[key].astype(np.float32)
            break
    else:
        keys = [k for k in mat if not k.startswith('_')]
        img  = mat[keys[0]].astype(np.float32)
    # Try common key names for ground truth
    gt = None
    for key in ['map', 'Map', 'gt', 'GT', 'groundtruth', 'label']:
        if key in mat:
            gt = mat[key].astype(np.float32)
            break
    if img.ndim == 2:                         # (H, W*D) packed
        raise ValueError('2D mat not supported — check array layout')
    if img.shape[0] < img.shape[-1]:          # (D, H, W) → (H, W, D)
        img = img.transpose(1, 2, 0)
    # Normalise per-band to [0, 1]
    mn, mx = img.min(axis=(0,1), keepdims=True), img.max(axis=(0,1), keepdims=True)
    img = (img - mn) / (mx - mn + 1e-8)
    print(f'  image: {img.shape}, gt: {gt.shape if gt is not None else None}')
    return img, gt


class TorchPCA:
    """Pure-torch PCA — avoids sklearn/numpy ABI issues."""
    def __init__(self, n_components):
        self.k = n_components
        self._components = None
        self._mean       = None

    def fit(self, pixels: torch.Tensor):            # (N, D)
        self._mean = pixels.mean(dim=0)             # (D,)
        X = pixels - self._mean
        cov = X.T @ X / (X.shape[0] - 1)           # (D, D)
        _, vecs = torch.linalg.eigh(cov)            # ascending eigenvalues
        self._components = vecs[:, -self.k:].flip(-1)  # (D, k) top-k
        return self

    def transform(self, image: torch.Tensor) -> torch.Tensor:   # (H, W, D)
        H, W, D = image.shape
        x = image.reshape(-1, D) - self._mean
        return (x @ self._components).reshape(H, W, self.k)

    def transform_vector(self, v: torch.Tensor) -> torch.Tensor:  # (D,)
        return (v - self._mean) @ self._components  # (k,)


def to_tensor(arr):
    """numpy array → torch float32 tensor, safe on NumPy 2.x."""
    return torch.tensor(arr.tolist(), dtype=torch.float32)


def plot_results(train_image, test_image, test_gt, scores, history, title, save_path=None):
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle(title, fontsize=13)

    # False-colour composite of test image (bands 0,1,2)
    rgb = test_image[:, :, :3].numpy()
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)
    axes[0].imshow(rgb); axes[0].set_title('Test image (RGB)'); axes[0].axis('off')

    # Ground truth
    axes[1].imshow(test_gt.numpy(), cmap='gray')
    axes[1].set_title('Ground truth'); axes[1].axis('off')

    # Detection map
    axes[2].imshow(scores.numpy(), cmap='hot')
    axes[2].set_title('THANTD scores'); axes[2].axis('off')

    # Training loss curve
    epochs     = [m['epoch'] for m in history]
    total_loss = [m['loss']  for m in history]
    dual_loss  = [m['loss_dual'] for m in history]
    axes[3].plot(epochs, total_loss, label='Total')
    axes[3].plot(epochs, dual_loss,  label='Dual triplet', linestyle='--')
    axes[3].set_xlabel('Epoch'); axes[3].set_ylabel('Loss')
    axes[3].set_title('Training loss'); axes[3].legend()
    axes[3].grid(True, alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved plot → {save_path}')
    plt.show()


print('Utilities ready.')

## 6 · Main loop: synthesise → train → evaluate

In [ ]:
all_results = {}

for mat_file in DATASET_FILES:
    dataset_name = mat_file.replace('.mat', '')
    print(f'\n{"="*60}')
    print(f'Dataset: {dataset_name}')
    print('='*60)

    # ── 6.1 Load real dataset ─────────────────────────────────────────────
    mat_path    = os.path.join(DATASET_DIR, mat_file)
    img_np, gt_np = load_mat(mat_path)
    image_real  = to_tensor(img_np)   # (H, W, D)
    gt_real     = to_tensor(gt_np) if gt_np is not None else None

    D = image_real.shape[-1]
    print(f'  Loaded: {image_real.shape}, bands={D}')

    # ── 6.2 Synthesise (500 × 250) image from real dataset stats ──────────
    print('Fitting SynthesisEngine to real dataset …')
    engine = SynthesisEngine.from_image(
        image_real,
        n_components=N_GMM_COMPS,
        gt=gt_real,
        device=device,
    )
    print('Generating synthetic image (500 × 250) …')
    synth_image, synth_gt = engine.generate(SYNTH_H, SYNTH_W, seed=42)
    # synth_image: (500, 250, D) float tensor
    # synth_gt:    (500, 250)   binary tensor
    print(f'  Synthetic image: {synth_image.shape},  targets: {synth_gt.sum().item():.0f} pixels')

    # ── 6.3 Save synthetic images to Drive ────────────────────────────────
    synth_dir = os.path.join(OUTPUT_DIR, dataset_name)
    os.makedirs(synth_dir, exist_ok=True)

    torch.save(
        {'image': synth_image, 'gt': synth_gt},
        os.path.join(synth_dir, 'synthetic_full.pt'),
    )
    print(f'  Saved synthetic image → {synth_dir}/synthetic_full.pt')

    # ── 6.4 Train / test split ────────────────────────────────────────────
    train_image = synth_image[:TRAIN_H]   # (350, 250, D)
    train_gt    = synth_gt[:TRAIN_H]      # (350, 250)
    test_image  = synth_image[TRAIN_H:]   # (150, 250, D)
    test_gt     = synth_gt[TRAIN_H:]      # (150, 250)

    torch.save({'image': train_image, 'gt': train_gt}, os.path.join(synth_dir, 'train.pt'))
    torch.save({'image': test_image,  'gt': test_gt},  os.path.join(synth_dir, 'test.pt'))
    print(f'  Train targets: {train_gt.sum().item():.0f}   Test targets: {test_gt.sum().item():.0f}')

    # ── 6.5 Optional PCA dimensionality reduction ─────────────────────────
    n_bands = D
    pca     = None
    if PCA_COMPONENTS is not None and D > PCA_COMPONENTS:
        print(f'Applying PCA: {D} → {PCA_COMPONENTS} bands …')
        pca = TorchPCA(PCA_COMPONENTS)
        pca.fit(train_image.reshape(-1, D))
        train_image = pca.transform(train_image)
        test_image  = pca.transform(test_image)
        n_bands     = PCA_COMPONENTS
        print(f'  PCA done: train {train_image.shape}, test {test_image.shape}')

    # ── 6.6 Target spectrum from training ground truth ────────────────────
    if train_gt.sum() > 0:
        tgt_pixels = train_image.reshape(-1, n_bands)[train_gt.reshape(-1) == 1]
        prior      = tgt_pixels.mean(dim=0)   # (n_bands,)
    else:
        raise RuntimeError('No target pixels in training split!')
    print(f'  Prior spectrum estimated from {tgt_pixels.shape[0]} target pixels')

    # ── 6.7 Build TripletDataset ──────────────────────────────────────────
    print('Building TripletDataset …')
    trip_dataset = build_dataset_from_image(
        train_image, train_gt, prior,
        n_triplets=TRAIN_CFG['n_triplets'],
        mu_max=0.1,
    )
    print(f'  Dataset size: {len(trip_dataset)} triplets')

    # ── 6.8 Build and train THANTD ────────────────────────────────────────
    model = THANTD(n_bands=n_bands, **MODEL_CFG)
    n_params = sum(p.numel() for p in model.parameters())
    print(f'THANTD  |  bands={n_bands}  d={MODEL_CFG["d_model"]}  '
          f'layers={MODEL_CFG["n_layers"]}  params={n_params:,}')

    ckpt_path = os.path.join(synth_dir, 'thantd_best.pt')
    print('Training …')
    history = train_thantd(
        model,
        trip_dataset,
        checkpoint_path=ckpt_path,
        **TRAIN_CFG,
    )
    print(f'Training done. Best checkpoint → {ckpt_path}')

    # Also save final state
    torch.save(model.state_dict(), os.path.join(synth_dir, 'thantd_final.pt'))

    # ── 6.9 Evaluate on test split ────────────────────────────────────────
    print('Evaluating on test split …')
    scores = thantd_detect(model, test_image.to(device), prior.to(device),
                            normalise=True)   # (150, 250)

    auc = roc_auc(scores.reshape(-1), test_gt.reshape(-1)).item()
    print(f'  AUC = {auc:.4f}')

    # ── 6.10 Save scores and history ─────────────────────────────────────
    torch.save(scores, os.path.join(synth_dir, 'test_scores.pt'))
    with open(os.path.join(synth_dir, 'history.json'), 'w') as f:
        json.dump(history, f, indent=2)

    all_results[dataset_name] = {'auc': auc, 'history': history}

    # ── 6.11 Plot ──────────────────────────────────────────────────────────
    plot_results(
        train_image.cpu(), test_image.cpu(), test_gt.cpu(), scores.cpu(),
        history,
        title=f'THANTD — {dataset_name}  (AUC={auc:.4f})',
        save_path=os.path.join(synth_dir, 'results.png'),
    )

print('\n' + '='*60)
print('ALL DONE')
for name, res in all_results.items():
    print(f'  {name:25s}  AUC = {res["auc"]:.4f}')

## 7 · Inspect a single result in more detail

In [ ]:
# Change this to whichever dataset you want to inspect
INSPECT = 'Sandiego'

synth_dir   = os.path.join(OUTPUT_DIR, INSPECT)
test_data   = torch.load(os.path.join(synth_dir, 'test.pt'))
test_scores = torch.load(os.path.join(synth_dir, 'test_scores.pt'))
history     = json.load(open(os.path.join(synth_dir, 'history.json')))

test_image_v = test_data['image']
test_gt_v    = test_data['gt']

# ROC curve
scores_flat = test_scores.reshape(-1)
gt_flat     = test_gt_v.reshape(-1)
thresholds  = torch.linspace(0, 1, 200)
pd_list, pf_list = [], []
for tau in thresholds:
    pred = (scores_flat >= tau).float()
    tp   = (pred * gt_flat).sum()
    fp   = (pred * (1 - gt_flat)).sum()
    fn   = ((1 - pred) * gt_flat).sum()
    tn   = ((1 - pred) * (1 - gt_flat)).sum()
    pd_list.append((tp / (tp + fn + 1e-8)).item())
    pf_list.append((fp / (fp + tn + 1e-8)).item())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f'THANTD — {INSPECT}', fontsize=13)

axes[0].plot(pf_list, pd_list)
axes[0].plot([0,1],[0,1],'k--', alpha=0.4)
axes[0].set_xlabel('False Alarm Rate'); axes[0].set_ylabel('Detection Rate')
axes[0].set_title(f'ROC  (AUC={all_results[INSPECT]["auc"]:.4f})')
axes[0].grid(True, alpha=0.3)

axes[1].imshow(test_scores.numpy(), cmap='hot')
axes[1].set_title('Detection scores'); axes[1].axis('off')

axes[2].imshow(test_gt_v.numpy(), cmap='gray')
axes[2].set_title('Ground truth'); axes[2].axis('off')

plt.tight_layout(); plt.show()

## 8 · Load a saved checkpoint and re-run inference

In [ ]:
# Example: reload best checkpoint for Sandiego
INSPECT    = 'Sandiego'
synth_dir  = os.path.join(OUTPUT_DIR, INSPECT)
ckpt       = torch.load(os.path.join(synth_dir, 'thantd_best.pt'), map_location=device)

# Recreate model with same config
model_loaded = THANTD(
    n_bands = ckpt.get('n_bands', PCA_COMPONENTS or D),
    **MODEL_CFG,
)
model_loaded.load_state_dict(ckpt['model_state'])
model_loaded.to(device).eval()
print(f'Loaded checkpoint from epoch {ckpt["epoch"]}  (loss={ckpt["loss"]:.4f})')

# Re-run inference
test_data   = torch.load(os.path.join(synth_dir, 'test.pt'))
test_img_r  = test_data['image']
test_gt_r   = test_data['gt']
# Re-derive prior from training data
train_data  = torch.load(os.path.join(synth_dir, 'train.pt'))
train_img_r = train_data['image']
train_gt_r  = train_data['gt']
prior_r     = train_img_r.reshape(-1, train_img_r.shape[-1])[train_gt_r.reshape(-1)==1].mean(0)

scores_r = thantd_detect(model_loaded, test_img_r.to(device), prior_r.to(device), normalise=True)
auc_r    = roc_auc(scores_r.reshape(-1), test_gt_r.reshape(-1)).item()
print(f'Reloaded model AUC = {auc_r:.4f}')